# Representation Learning Assignment
This assignment covers key topics in representation learning using PyTorch datasets. You will implement tasks for Contrastive Learning, Energy-Based Models. Use torchvision.datasets.MNIST for Exercises 1, 2.

**Total Points: 12**
- Exercise 1: Contrastive Learning (8 points)
- Exercise 2: Energy-Based Models (4 points)

Import necessary libraries and load datasets as needed.


In [1]:
# --- Install dependencies (CPU-only build by default) ---
!pip install torch torchvision --extra-index-url https://download.pytorch.org/whl/cpu
!pip install -q torch_geometric

# --- Imports ---
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
from torchvision import datasets, transforms
from torch.utils.data import Subset, DataLoader

print("Torch:", torch.__version__, "Torchvision:", torchvision.__version__)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Looking in indexes: https://pypi.org/simple, https://download.pytorch.org/whl/cpu
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 8.5 MB/s eta 0:00:000m eta 0:00:01:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.9/183.9 MB 3.7 MB/s eta 0:00:000m eta 0:00:010:00:01
  Attempting uninstall: torch
    Found existing installation: torch 2.7.1
    Uninstalling torch-2.7.1:
      Successfully uninstalled torch-2.7.1
Torch: 2.8.0+cpu Torchvision: 0.23.0+cpu
Using device: cpu


In [ ]:
# --- MNIST (for Exercises 1, 2) ---
mnist_transform = transforms.ToTensor()

mnist_train_full = datasets.MNIST(
    root="./data", train=True, download=True, transform=mnist_transform
)
mnist_test_full = datasets.MNIST(
    root="./data", train=False, download=True, transform=mnist_transform
)

# Smaller subsets for quicker experiments
train_indices = list(range(1000))
test_indices = list(range(200))

mnist_train = Subset(mnist_train_full, train_indices)
mnist_test = Subset(mnist_test_full, test_indices)

print(f"MNIST Train Subset: {len(mnist_train)} samples")
print(f"MNIST Test Subset: {len(mnist_test)} samples")

## Exercise 1: Contrastive Learning (8 points)

We’ll simulate contrastive learning on MNIST images by computing similarities, pairs, and losses.

### Task 1a (1 point): Cosine Similarity

Compute row-wise cosine similarity between two batches of embeddings.

In [ ]:
### Ex-1-Task-1
import torch
import torch.nn.functional as F

def compute_similarity(emb1, emb2):
    """
    emb1, emb2: (B, D)
    Returns: (B,) cosine similarities
    """
    ### BEGIN SOLUTION 
    # YOUR CODE HERE
    # Compute row-wise cosine similarity
    return F.cosine_similarity(emb1, emb2, dim=1)
    ### END SOLUTION

# Test
x = torch.randn(5, 10)
y = x.clone()
print(compute_similarity(x, y))

tensor([1.0000, 1.0000, 1.0000, 1.0000, 1.0000])


In [3]:
# INTENTIONALLY LEFT BLANK

### Task 1b: Positive & Negative Pairs (2 points)

Generate positive pairs (same sample) and negative pairs (different sample) for a batch.
Simulate negatives by randomly shuffling batch.


In [4]:
### Ex-1-Task-2
def generate_pairs(batch):
    """
    batch: (B, D)
    Returns:
        pos: (B, D)
        neg: (B, D)
    """
    ### BEGIN SOLUTION 
    # YOUR CODE HERE

    # Positive pairs(same sample)
    pos = batch.clone()
    # Negative pairs
    idx = torch.randperm(batch.size(0))
    neg = batch[idx]
    return pos, neg
    ### END SOLUTION

# Test
batch = torch.randn(5, 8)
pos, neg = generate_pairs(batch)
print("Positive pairs:\n", pos)
print("Negative pairs:\n", neg)

Positive pairs:
 tensor([[ 0.1176,  0.4396, -0.2588, -0.6557,  0.2702, -0.3622, -0.4160, -1.4513],
        [ 0.2128,  0.6034, -0.0880,  0.9895, -0.7477, -0.1997,  0.1275,  0.6238],
        [ 0.3726, -0.9230,  0.7673,  0.8789,  0.9815, -0.5945, -2.0380, -0.7937],
        [ 0.8226, -0.7716,  2.0983, -0.5201, -0.0215, -0.9026, -0.3043,  0.3621],
        [-0.1175,  1.5259,  1.5033,  0.8075, -0.9506,  0.4633,  0.1521, -0.9498]])
Negative pairs:
 tensor([[ 0.2128,  0.6034, -0.0880,  0.9895, -0.7477, -0.1997,  0.1275,  0.6238],
        [-0.1175,  1.5259,  1.5033,  0.8075, -0.9506,  0.4633,  0.1521, -0.9498],
        [ 0.1176,  0.4396, -0.2588, -0.6557,  0.2702, -0.3622, -0.4160, -1.4513],
        [ 0.8226, -0.7716,  2.0983, -0.5201, -0.0215, -0.9026, -0.3043,  0.3621],
        [ 0.3726, -0.9230,  0.7673,  0.8789,  0.9815, -0.5945, -2.0380, -0.7937]])


In [5]:
# INTENTIONALLY LEFT BLANK

### Task 1c:NT-Xent Loss (2 points)

Implement normalized temperature-scaled cross-entropy loss for a batch.

In [6]:
### Ex-1-Task-3
import torch
import torch.nn.functional as F

def nt_xent_loss(sims, temp=0.5):
    """
    sims: (B, B) similarity matrix
    temp: scalar temperature
    Returns: scalar loss
    """
    ### BEGIN SOLUTION 
    # YOUR CODE HERE
    # For each row, the positive is the diagonal element
    B = sims.size(0)
    logits = sims / temp
    labels = torch.arange(B, device=logits.device)
    loss = F.cross_entropy(logits, labels)
    return loss    
    ### END SOLUTION

# Test
sims = torch.randn(3, 3)
print("NT-Xent loss:", nt_xent_loss(sims))

NT-Xent loss: tensor(1.5704)


In [7]:
# Quick visible test with identity similarity
sims = torch.eye(4)  # 4x4 identity matrix
loss = nt_xent_loss(sims)
print("NT-Xent loss:", loss.item())  # Should be > 0


NT-Xent loss: 0.3407530188560486


In [8]:
# INTENTIONALLY LEFT BLANK


### Task 1d: Augment MNIST Image (2 points)

Apply random small rotation ±10° to a single MNIST image.


In [13]:
### Ex-1-Task-4
from torchvision import transforms

augment = transforms.RandomRotation(degrees=10)

def augment_image(img):
    """
    img: (28, 28) tensor
    Returns: augmented image (28,28)
    """
    ### BEGIN SOLUTION 
    # YOUR CODE HERE
   # torchvision transforms expect (C, H, W), so add channel if needed
    if img.dim() == 2:
        img = img.unsqueeze(0)
    aug_img = augment(img)
    return aug_img.squeeze(0)
    ### END SOLUTION

In [ ]:
img, label = mnist_train[0]
aug_img = augment_image(img)
print("Original shape:", img.shape)
print("Augmented shape:", aug_img.shape)

In [15]:
# INTENTIONALLY LEFT BLANK

### Task 1e: Contrastive Embedding Distance (1 point)

Compute Euclidean distance between two batches of embeddings row-wise.

In [ ]:
### Ex-1-Task-5
import torch

def embedding_distance(emb1, emb2):
    """
    emb1, emb2: (B, D)
    Returns: (B,) Euclidean distances
    """
    ### BEGIN SOLUTION 
    # YOUR CODE HERE
    return torch.norm(emb1 - emb2, dim=1)

    ### END SOLUTION

# --- Test Example ---
batch_size, dim = 4, 5
emb1 = torch.randn(batch_size, dim)
emb2 = torch.randn(batch_size, dim)

distances = embedding_distance(emb1, emb2)
print("Embeddings 1:\n", emb1)
print("Embeddings 2:\n", emb2)
print("Row-wise Euclidean distances:\n", distances)

In [ ]:
# INTENTIONALLY LEFT BLANK

## Exercise 2: Energy-Based Models (4 points)



### Task 2a (2 points): Define Energy Function

Energy function for binary classification: E(x,y;w)=−y⋅(w⋅x)

In [16]:
### Ex-2-Task-1
def energy_function(x, y, w):
    """
    x: (784,), y: +1/-1, w: (784,)
    Returns: scalar energy
    """
    ### BEGIN SOLUTION 
    # YOUR CODE HERE
    return -y * torch.dot(w, x)
    ### END SOLUTION

# Test
x = torch.randn(784)
w = torch.randn(784)
y = 1
print("Energy:", energy_function(x, y, w))

Energy: tensor(17.0920)


In [17]:
# INTENTIONALLY LEFT BLANK

### Task 2b (2 points): Perceptron Loss (2 point)

Compute perceptron loss: max(0, E(x, y_true) - E(x, y_pred))

In [18]:
### Ex-2-Task-2
def perceptron_loss(x, y_true, y_pred, w):
    """
    x: (784,), y_true, y_pred: +1/-1
    w: (784,)
    Returns: scalar loss
    """
    ### BEGIN SOLUTION 
    # YOUR CODE HERE
    # Compute energy for true and predicted labels
    E_true = energy_function(x, y_true, w)
    E_pred = energy_function(x, y_pred, w)
    return torch.clamp(E_true - E_pred, min=0)
    ### END SOLUTION

# Test
print("Perceptron loss:", perceptron_loss(x, 1, -1, w))

Perceptron loss: tensor(34.1840)


In [19]:
# INTENTIONALLY LEFT BLANK